In [4]:
import os
import sys
import glob

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from pycaret.clustering import setup, create_model, assign_model, models, pull
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from datetime import datetime
import warnings
import time

from utils import preprocessing

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [8]:
warnings.filterwarnings('ignore')
DATA_PATH = '../data'
OUTPUT_DIR = '../results'
IMAGE_DIR = '../images'
    
document_df = preprocessing.get_default_data()
    


📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1266.9


In [9]:
# TF-IDF 벡터화
print("🔤 TF-IDF 벡터화 중...")
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    min_df=2,
    max_df=0.8
)
X = vectorizer.fit_transform(document_df['processed_text'].astype(str))
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print(f"✅ TF-IDF 행렬 생성: {tfidf_df.shape}\n")
    
# 클러스터 수 추정
n_clusters = max(2, int(np.sqrt(len(document_df))))
print(f"🎯 추정 클러스터 수: {n_clusters}\n")
    
# 디렉토리 생성
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
    
# =============================================================================
# PyCaret Setup
# =============================================================================
print(f"{'='*70}")
print("⚙️  PyCaret 환경 설정 중...")
print(f"{'='*70}\n")
    
s = setup(
    data=tfidf_df,
    session_id=42,
    verbose=False,
    html=False,
    system_log=False
    )
    
print("✅ PyCaret 설정 완료\n")
    
# 사용 가능한 모델 확인
print("📋 사용 가능한 모델:")
available_models = models()
print(available_models)
print()
    


🔤 TF-IDF 벡터화 중...
✅ TF-IDF 행렬 생성: (51, 3203)

🎯 추정 클러스터 수: 7

⚙️  PyCaret 환경 설정 중...

✅ PyCaret 설정 완료

📋 사용 가능한 모델:
                                       Name  \
ID                                            
kmeans                   K-Means Clustering   
ap                     Affinity Propagation   
meanshift             Mean Shift Clustering   
sc                      Spectral Clustering   
hclust             Agglomerative Clustering   
dbscan     Density-Based Spatial Clustering   
optics                    OPTICS Clustering   
birch                      Birch Clustering   
kmodes                   K-Modes Clustering   

                                                   Reference  
ID                                                            
kmeans                        sklearn.cluster._kmeans.KMeans  
ap         sklearn.cluster._affinity_propagation.Affinity...  
meanshift              sklearn.cluster._mean_shift.MeanShift  
sc              sklearn.cluster._spectral.SpectralC

In [10]:
# =============================================================================
# 클러스터링 알고리즘 정의 및 실행
# =============================================================================
print(f"{'='*70}")
print("🤖 클러스터링 알고리즘 실행 중...")
print(f"{'='*70}\n")
    
# PyCaret 모델 ID 매핑
model_configs = {
    'kmeans': {'name': 'K-Means', 'num_clusters': n_clusters},
    'ap': {'name': 'Affinity Propagation', 'num_clusters': None},
    'meanshift': {'name': 'Mean Shift', 'num_clusters': None},
    'sc': {'name': 'Spectral Clustering', 'num_clusters': n_clusters},
    'hclust': {'name': 'Agglomerative Clustering', 'num_clusters': n_clusters},
    'dbscan': {'name': 'DBSCAN', 'num_clusters': None},
    'optics': {'name': 'OPTICS', 'num_clusters': None},
    'birch': {'name': 'Birch', 'num_clusters': n_clusters},
    'kmodes': {'name': 'K-Modes', 'num_clusters': n_clusters}
}
    
results = []
trained_models = {}
cluster_labels_dict = {}
    
for model_id, config in model_configs.items():
    model_name = config['name']
    num_clusters = config['num_clusters']
        
    print(f"   ⏳ {model_name} ({model_id}) 실행 중...", end=' ')
    start_time = time.time()
        
    try:
        # 모델 생성
        if num_clusters is not None:
            model = create_model(model_id, num_clusters=num_clusters, verbose=False)
        else:
            model = create_model(model_id, verbose=False)
            
        elapsed_time = time.time() - start_time
            
        # 클러스터 할당
        clustered_data = assign_model(model)
        labels_raw = clustered_data['Cluster'].values
            
        # 'Cluster X' 형태의 문자열을 숫자로 변환
        labels = np.array([
            int(str(label).replace('Cluster ', '')) if 'Cluster' in str(label) else label
            for label in labels_raw
        ])
            
        # 성능 메트릭 가져오기
        metrics_df = pull()
            
        # 클러스터 정보
        unique_labels = set(labels)
        n_clusters_found = len(unique_labels)
        n_noise = list(labels).count(-1) if -1 in unique_labels else 0
            
        # 결과 저장
        result = {
            'Model ID': model_id,
            'Algorithm': model_name,
            'Silhouette': metrics_df['Silhouette'].values[0] if 'Silhouette' in metrics_df.columns else None,
            'Calinski-Harabasz': metrics_df['Calinski-Harabasz'].values[0] if 'Calinski-Harabasz' in metrics_df.columns else None,
            'Davies-Bouldin': metrics_df['Davies-Bouldin'].values[0] if 'Davies-Bouldin' in metrics_df.columns else None,
            'N_Clusters': n_clusters_found,
            'N_Noise': n_noise,
            'Time (s)': elapsed_time
        }
            
        results.append(result)
        trained_models[model_id] = model
        cluster_labels_dict[model_name] = labels
            
        print(f"✅ ({elapsed_time:.2f}s, {n_clusters_found} clusters, {n_noise} noise)")
            
    except Exception as e:
        elapsed_time = time.time() - start_time
        print(f"❌ Error: {str(e)[:50]}")
        results.append({
            'Model ID': model_id,
            'Algorithm': model_name,
            'Silhouette': None,
            'Calinski-Harabasz': None,
            'Davies-Bouldin': None,
            'N_Clusters': None,
            'N_Noise': None,
            'Time (s)': elapsed_time
        })
    
# 결과 DataFrame
results_df = pd.DataFrame(results)
    


🤖 클러스터링 알고리즘 실행 중...

   ⏳ K-Means (kmeans) 실행 중... ✅ (1.80s, 7 clusters, 0 noise)
   ⏳ Affinity Propagation (ap) 실행 중... ✅ (1.44s, 17 clusters, 0 noise)
   ⏳ Mean Shift (meanshift) 실행 중... ✅ (4.57s, 1 clusters, 0 noise)
   ⏳ Spectral Clustering (sc) 실행 중... ✅ (1.46s, 7 clusters, 0 noise)
   ⏳ Agglomerative Clustering (hclust) 실행 중... ✅ (1.38s, 7 clusters, 0 noise)
   ⏳ DBSCAN (dbscan) 실행 중... ✅ (1.41s, 1 clusters, 51 noise)
   ⏳ OPTICS (optics) 실행 중... ✅ (1.62s, 2 clusters, 37 noise)
   ⏳ Birch (birch) 실행 중... ✅ (1.40s, 7 clusters, 0 noise)
   ⏳ K-Modes (kmodes) 실행 중... ✅ (2.46s, 7 clusters, 0 noise)


In [ ]:
# =============================================================================
# 결과 저장 (텍스트)
# =============================================================================
print(f"\n{'='*70}")
print("💾 결과 저장 중...")
print(f"{'='*70}\n")
    
results_path = os.path.join(OUTPUT_DIR, f"ejm_pycaret_clustering_{timestamp}.txt")
with open(results_path, "w", encoding="utf-8") as f:
    f.write("="*70 + "\n")
    f.write("PyCaret 3.3.2 - 9가지 클러스터링 알고리즘 비교 결과\n")
    f.write("="*70 + "\n\n")
    f.write(f"분석 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"문서 수: {len(document_df)}\n")
    f.write(f"TF-IDF 특성 수: {tfidf_df.shape[1]}\n")
    f.write(f"목표 클러스터 수: {n_clusters}\n\n")
        
    f.write("="*70 + "\n")
    f.write("📊 평가 지표 비교\n")
    f.write("="*70 + "\n\n")
    f.write(results_df.to_string(index=False))
    f.write("\n\n")
        
    f.write("="*70 + "\n")
    f.write("📝 평가 지표 설명\n")
    f.write("="*70 + "\n")
    f.write("• Silhouette Score: -1~1, 높을수록 좋음 (클러스터 분리도)\n")
    f.write("• Calinski-Harabasz: 높을수록 좋음 (클러스터 밀집도)\n")
    f.write("• Davies-Bouldin: 낮을수록 좋음 (클러스터 간 유사도)\n")
    f.write("• N_Noise: 노이즈로 분류된 데이터 포인트 수\n")
        
    # Best 모델 선정
    valid_results = results_df[results_df['Silhouette'].notna()]
    if len(valid_results) > 0:
        best_silhouette = valid_results.loc[valid_results['Silhouette'].idxmax()]
        f.write(f"\n{'='*70}\n")
        f.write("🏆 Best Model (Silhouette Score 기준)\n")
        f.write(f"{'='*70}\n")
        f.write(f"Algorithm: {best_silhouette['Algorithm']}\n")
        f.write(f"Model ID: {best_silhouette['Model ID']}\n")
        f.write(f"Silhouette Score: {best_silhouette['Silhouette']:.4f}\n")
        f.write(f"Calinski-Harabasz: {best_silhouette['Calinski-Harabasz']:.4f}\n")
        f.write(f"Davies-Bouldin: {best_silhouette['Davies-Bouldin']:.4f}\n")
    
print(f"✅ 텍스트 결과 저장: {results_path}")
    
# CSV 저장
csv_path = os.path.join(OUTPUT_DIR, f"ejm_pycaret_results_{timestamp}.csv")
results_df.to_csv(csv_path, index=False)
print(f"✅ CSV 결과 저장: {csv_path}")
    
# =============================================================================
# 시각화
# =============================================================================
print(f"\n{'='*70}")
print("📈 시각화 생성 중...")
print(f"{'='*70}\n")
    
# 1. Silhouette Score 비교
valid_results = results_df[results_df['Silhouette'].notna()].sort_values('Silhouette', ascending=False)
    
if len(valid_results) > 0:
    plt.figure(figsize=(14, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(valid_results)))
    bars = plt.bar(range(len(valid_results)), valid_results['Silhouette'], color=colors)
    plt.xticks(range(len(valid_results)), valid_results['Algorithm'], rotation=45, ha='right')
    plt.ylabel("Silhouette Score", fontsize=12)
    plt.title("PyCaret Clustering Algorithms - Silhouette Score Comparison", 
                 fontsize=14, fontweight='bold')
    plt.axhline(y=0, color='r', linestyle='--', linewidth=0.5, alpha=0.7)
    plt.grid(axis='y', alpha=0.3)
        
    # 값 표시
    for i, (bar, val) in enumerate(zip(bars, valid_results['Silhouette'])):
        plt.text(bar.get_x() + bar.get_width()/2, val, f'{val:.3f}',
                    ha='center', va='bottom' if val > 0 else 'top', fontsize=9)
        
    plt.tight_layout()
    silhouette_path = os.path.join(IMAGE_DIR, f"pycaret_silhouette_{timestamp}.png")
    plt.savefig(silhouette_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Silhouette 비교: {silhouette_path}")
    
# 2. 실행 시간 비교
valid_time = results_df[results_df['Time (s)'].notna()].sort_values('Time (s)')
    
if len(valid_time) > 0:
    plt.figure(figsize=(14, 6))
    colors = plt.cm.coolwarm(np.linspace(0, 1, len(valid_time)))
    bars = plt.bar(range(len(valid_time)), valid_time['Time (s)'], color=colors)
    plt.xticks(range(len(valid_time)), valid_time['Algorithm'], rotation=45, ha='right')
    plt.ylabel("Time (seconds)", fontsize=12)
    plt.title("PyCaret Clustering Algorithms - Execution Time", 
                 fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
        
    # 값 표시
    for bar, val in zip(bars, valid_time['Time (s)']):
        plt.text(bar.get_x() + bar.get_width()/2, val, f'{val:.2f}s',
                    ha='center', va='bottom', fontsize=9)
        
    plt.tight_layout()
    time_path = os.path.join(IMAGE_DIR, f"pycaret_time_{timestamp}.png")
    plt.savefig(time_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 실행 시간 비교: {time_path}")
    
# 3. 종합 비교 (히트맵)
comparison_data = results_df[['Algorithm', 'Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin']].copy()
comparison_data = comparison_data.dropna()
    
if len(comparison_data) > 0:
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
        
    # Davies-Bouldin은 낮을수록 좋으므로 역전
    comparison_data['Davies-Bouldin'] = -comparison_data['Davies-Bouldin']
        
    metrics = ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin']
    normalized = scaler.fit_transform(comparison_data[metrics])
        
    plt.figure(figsize=(12, 8))
    im = plt.imshow(normalized.T, cmap='RdYlGn', aspect='auto')
    plt.colorbar(im, label='Normalized Score (0-1)')
    plt.yticks(range(len(metrics)), ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin (inverted)'])
    plt.xticks(range(len(comparison_data)), comparison_data['Algorithm'], rotation=45, ha='right')
    plt.title("PyCaret Algorithms Performance Heatmap (Higher is Better)", 
                 fontsize=14, fontweight='bold')
        
    # 값 표시
    for i in range(len(metrics)):
        for j in range(len(comparison_data)):
            text_color = 'white' if normalized[j, i] < 0.5 else 'black'
            plt.text(j, i, f'{normalized[j, i]:.2f}', 
                        ha='center', va='center', color=text_color, fontsize=9)
        
    plt.tight_layout()
    heatmap_path = os.path.join(IMAGE_DIR, f"pycaret_heatmap_{timestamp}.png")
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 성능 히트맵: {heatmap_path}")
    
# 4. PCA 시각화 (상위 3개 알고리즘)
print("\n   🔍 PCA 시각화 생성 중...")
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(tfidf_df)
    
top_3 = valid_results.head(3)['Algorithm'].tolist()
    
fig, axes = plt.subplots(1, min(3, len(top_3)), figsize=(6*min(3, len(top_3)), 5))
if len(top_3) == 1:
    axes = [axes]
elif len(top_3) == 0:
    print("   ⚠️  유효한 결과가 없어 PCA 시각화를 건너뜁니다.")
else:
    for idx, algo_name in enumerate(top_3[:3]):
        if algo_name in cluster_labels_dict:
            labels = cluster_labels_dict[algo_name]
            ax = axes[idx] if len(top_3) > 1 else axes[0]
                
            # 레이블이 숫자형인지 확인
            try:
                labels_numeric = labels.astype(float)
            except:
                # 문자열 레이블을 숫자로 변환
                unique_labels = sorted(set(labels))
                label_map = {label: i for i, label in enumerate(unique_labels)}
                labels_numeric = np.array([label_map[label] for label in labels])
                
            scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_numeric, 
                                   cmap='tab10', alpha=0.6, edgecolors='w', linewidth=0.5, s=50)
            score = valid_results[valid_results['Algorithm']==algo_name]['Silhouette'].values[0]
            ax.set_title(f"{algo_name}\n(Score: {score:.3f})",
                            fontsize=12, fontweight='bold')
            ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%})")
            ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%})")
            plt.colorbar(scatter, ax=ax, label='Cluster')
        
    plt.suptitle("Top 3 PyCaret Clustering Results (PCA Visualization)", 
                    fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    pca_path = os.path.join(IMAGE_DIR, f"pycaret_top3_pca_{timestamp}.png")
    plt.savefig(pca_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Top-3 PCA: {pca_path}")
    
# 5. 클러스터 수 비교
cluster_counts = results_df[['Algorithm', 'N_Clusters', 'N_Noise']].dropna()
    
if len(cluster_counts) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(cluster_counts))
    width = 0.35
        
    bars1 = ax.bar(x - width/2, cluster_counts['N_Clusters'], width, 
                      label='Clusters Found', color='skyblue')
    bars2 = ax.bar(x + width/2, cluster_counts['N_Noise'], width,
                      label='Noise Points', color='coral')
        
    ax.set_xlabel('Algorithm')
    ax.set_ylabel('Count')
    ax.set_title('Number of Clusters and Noise Points by Algorithm', 
                    fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(cluster_counts['Algorithm'], rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
        
    # 값 표시
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}', ha='center', va='bottom', fontsize=9)
        
    plt.tight_layout()
    clusters_path = os.path.join(IMAGE_DIR, f"pycaret_clusters_{timestamp}.png")
    plt.savefig(clusters_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"   ✅ 클러스터 수 비교: {clusters_path}")
    
# =============================================================================
# 최종 요약
# =============================================================================
print(f"\n{'='*70}")
print("✨ 분석 완료!")
print(f"{'='*70}")
print(f"\n📊 결과 요약:")
print(results_df.to_string(index=False))
    
if len(valid_results) > 0:
    best = valid_results.iloc[0]
    print(f"\n🏆 Best Algorithm: {best['Algorithm']} ({best['Model ID']})")
    print(f"   Silhouette Score: {best['Silhouette']:.4f}")
    print(f"   Clusters Found: {int(best['N_Clusters'])}")
    print(f"   Execution Time: {best['Time (s)']:.2f}s")
    
print(f"\n📁 모든 결과가 저장되었습니다:")
print(f"   - {results_path}")
print(f"   - {IMAGE_DIR}/")
print(f"{'='*70}\n")